In [2]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import io
import json
import numpy as np
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image as PILImage
from tensorflow.keras.preprocessing import image

# --- 1. CRITICAL GPU FIX ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled.")
    except RuntimeError as e:
        print(f"GPU config error: {e}")

# --- 2. CONFIGURATION ---
MODEL_PATH = "models/gpu_trained_model.keras"
CLASS_JSON_PATH = "models/class_indices.json"
# CHANGED: Reverted to 224x224 to match your saved model's brain!
IMAGE_SIZE = (224, 224)

# --- 3. LOAD AI ONCE ---
print("Loading model and classes (please wait a moment)...")
try:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    with open(CLASS_JSON_PATH, "r") as f:
        class_mapping = json.load(f)
    print("Ready! Model successfully loaded.")
except Exception as e:
    print(f"Error loading model: {e}")

# --- 4. THE CORE PREDICTION ENGINE ---
# --- 4. THE CORE PREDICTION ENGINE ---
def run_prediction_on_image(img):
    # This is your master list of all 14 original diseases
    MASTER_DISEASES = [
        "Actinic keratoses", "Basal cell carcinoma", "Benign keratosis",
        "Chickenpox", "Cowpox", "Dermatofibroma", "Healthy", "HFMD",
        "Measles", "Melanocytic nevi", "Melanoma", "Monkeypox",
        "Squamous cell carcinoma", "Vascular lesions"
    ]

    img = img.resize(IMAGE_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]

    # -- THE DOUBLE LOOKUP --
    best_index = np.argmax(predictions)
    confidence_score = predictions[best_index] * 100

    # Step 1: Get the folder number (e.g., "10")
    folder_number_str = class_mapping[str(best_index)]

    # Step 2: Use the folder number to get the real name from the master list
    predicted_disease = MASTER_DISEASES[int(folder_number_str)].upper()

    print("\n" + "=" * 45)
    print("                DIAGNOSIS                ")
    print("=" * 45)
    print(f"Condition : {predicted_disease}")
    print(f"Confidence: {confidence_score:.2f}%\n")

    print("--- Top 3 Possibilities ---")
    top_3_indices = np.argsort(predictions)[-3:][::-1]
    for i in top_3_indices:
        idx_str = str(i)
        if idx_str in class_mapping:
            folder_num = int(class_mapping[idx_str])
            real_name = MASTER_DISEASES[folder_num]
            score = predictions[i] * 100
            print(f"{real_name:<25}: {score:.2f}%")
    print("=" * 45)

# --- 5. INTERACTIVE UPLOAD BUTTON LOGIC ---
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output(wait=True)
        if not uploader.value:
            return

        print("Processing uploaded image...")
        try:
            if isinstance(uploader.value, dict):
                filename = list(uploader.value.keys())[0]
                content = uploader.value[filename]['content']
            else:
                content = uploader.value[0]['content']

            if isinstance(content, memoryview):
                content = content.tobytes()

            img_display = widgets.Image(value=content, format='jpg', width=300)
            display(img_display)

            img = PILImage.open(io.BytesIO(content)).convert('RGB')
            run_prediction_on_image(img)

        except Exception as e:
            print(f"\nError during processing: {str(e)}")

        uploader.value = ()

uploader.observe(on_upload_change, names='value')

# --- 6. DEFAULT TEST PATH LOGIC ---
def predict_local_file(file_path):
    print(f"\nProcessing default image: {file_path}")
    if os.path.exists(file_path):
        img = PILImage.open(file_path).convert('RGB')
        run_prediction_on_image(img)
    else:
        print(f"Error: The image file '{file_path}' does not exist.")

def convert_to_wsl_path(win_path):
    """Automatically translates a Windows C:\ path into a Linux /mnt/c/ path"""
    # Only convert if it actually looks like a Windows C: drive path
    if win_path.lower().startswith("c:\\"):
        win_path = "/mnt/c/" + win_path[3:]
    # Flip all Windows backslashes to Linux forward slashes
    return win_path.replace("\\", "/")

# --- 7. EXECUTION ---
display(widgets.VBox([widgets.Label("Upload a skin lesion image to test:"), uploader, output]))

if __name__ == "__main__":
    # You can now paste raw Windows paths here! The function will fix it for WSL.
    RAW_WINDOWS_PATH = r"C:\Users\mjvim\Downloads\measles.jpg"

    linux_ready_path = convert_to_wsl_path(RAW_WINDOWS_PATH)
    predict_local_file(linux_ready_path)

Loading model and classes (please wait a moment)...
Ready! Model successfully loaded.



Processing default image: /mnt/c/Users/mjvim/Downloads/measles.jpg

                DIAGNOSIS                
Condition : 6
Confidence: 77.23%

--- Top 3 Possibilities ---
6                        : 77.23%
1                        : 13.46%
2                        : 2.72%
